# Person 2 — Multilingual NLP & Retrieval Experiment
### BharatAssist — 3-Day Independent Module

**Scope for these 3 days:** build and test multilingual embedding + retrieval, now pointed at Person 1's **real 75-chunk dataset** (`data/bharatassist_chunks.json`, 15 real government schemes). The fake 8-scheme placeholder data has been removed — the real English-only content works fine against Hindi queries because the embedding model is multilingual.

In [1]:
# Run once per environment (Colab already has most of this)
!pip install -q sentence-transformers scikit-learn numpy


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Day 1 — Multilingual queries + embedding model selection

In [2]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\NLP\Multilingual_Retrieval\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Step 1: 20 test queries (10 English + 10 Hindi)**, loaded from `data/queries_hi_en.json` so the notebook stays in sync with the scripts — covering housing, health insurance, pensions, scholarships, LPG subsidy, skill training, maternity benefit, and business loans.

In [5]:
with open("queries_hi_en.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

print(f"{len(queries)} queries loaded ({sum(q['language']=='en' for q in queries)} EN, {sum(q['language']=='hi' for q in queries)} HI)")

20 queries loaded (10 EN, 10 HI)


**Step 2: compare two multilingual embedding models.** A same-meaning English/Hindi pair should score high similarity; an unrelated sentence should score low. Whichever model shows the bigger gap is the better pick for Day 2.

In [6]:
MODEL_CANDIDATES = [
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "intfloat/multilingual-e5-base",
]

english_sentence = "What is the eligibility for the housing scheme?"
hindi_sentence = "आवास योजना की पात्रता क्या है?"
unrelated_sentence = "What is the capital of France?"

for model_name in MODEL_CANDIDATES:
    print(f"\n=== {model_name} ===")
    model = SentenceTransformer(model_name)
    emb = model.encode([english_sentence, hindi_sentence, unrelated_sentence])
    sim_matching = cosine_similarity([emb[0]], [emb[1]])[0][0]
    sim_unrelated = cosine_similarity([emb[0]], [emb[2]])[0][0]
    print(f"dim={emb.shape[1]} | EN-HI (should be high): {sim_matching:.4f} | EN-unrelated (should be low): {sim_unrelated:.4f}")


=== sentence-transformers/paraphrase-multilingual-mpnet-base-v2 ===


d:\NLP\Multilingual_Retrieval\venv\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2129.05it/s]


dim=768 | EN-HI (should be high): 0.8821 | EN-unrelated (should be low): 0.1512

=== intfloat/multilingual-e5-base ===


d:\NLP\Multilingual_Retrieval\venv\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3178.85it/s]


dim=768 | EN-HI (should be high): 0.8923 | EN-unrelated (should be low): 0.7539


**Pick your model based on the results above**, then set it here — everything below uses this one variable. Keep this the same value as `MODEL_NAME` in `scripts/generate_embeddings.py` so the notebook and scripts agree.

In [7]:
CHOSEN_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # <- update after comparing, match generate_embeddings.py

model = SentenceTransformer(CHOSEN_MODEL)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7008.47it/s]


## Day 2 — Real scheme chunks + embedding experiment

Loading Person 1's real dataset: **75 chunks across 15 government schemes** (PM-KISAN, Ayushman Bharat, PMAY urban/rural, Ujjwala, PM-SVANidhi, PM Vishwakarma, Mudra Yojana, Atal Pension Yojana, PMSBY, PMJJBY, Sukanya Samriddhi, PMMVY, PMFBY, Stand-Up India). Each chunk has fields `chunk_id`, `scheme`, `section` (eligibility/benefits/age_income/coverage_category/application), and `text`. Content is English-only — the multilingual model handles Hindi queries against it directly, so no bilingual placeholder data is needed.

In [9]:
with open("bharatassist_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"{len(chunks)} chunks loaded across {len(set(c['scheme'] for c in chunks))} schemes")
print(chunks[0])  # sanity check the structure

75 chunks loaded across 15 schemes
{'chunk_id': '1_1', 'scheme': 'pradhan mantri kisan samman nidhi (pm-kisan)', 'section': 'eligibility', 'text': 'Scheme: pradhan mantri kisan samman nidhi (pm-kisan). Eligibility: landholding farmer families meeting scheme conditions institutional landholders, income-tax payers and specified higher-economic-status categories are excluded.'}


In [11]:
query_texts = [q["text"] for q in queries]
chunk_texts = [c["text"] for c in chunks]

query_embeddings = model.encode(query_texts, show_progress_bar=True)
chunk_embeddings = model.encode(chunk_texts, show_progress_bar=True)

print(f"query_embeddings shape: {query_embeddings.shape}")
print(f"chunk_embeddings shape: {chunk_embeddings.shape}")

np.save("data/query_embeddings.npy", query_embeddings)
np.save("data/scheme_embeddings.npy", chunk_embeddings)

Batches: 100%|██████████| 3/3 [00:07<00:00,  2.63s/it]

query_embeddings shape: (20, 768)
chunk_embeddings shape: (75, 768)


## Day 3 — Semantic similarity + Top-3 retrieval

In [12]:
def top_k_retrieval(query_embeddings, chunk_embeddings, k=3):
    sims = cosine_similarity(query_embeddings, chunk_embeddings)  # (n_queries, n_chunks)
    top_k_indices = np.argsort(-sims, axis=1)[:, :k]
    top_k_scores = np.take_along_axis(sims, top_k_indices, axis=1)
    return top_k_indices, top_k_scores

top_k_indices, top_k_scores = top_k_retrieval(query_embeddings, chunk_embeddings, k=3)

In [13]:
results = []
for i, q in enumerate(queries):
    matches = []
    print(f"\nQuery ({q['language']}) [{q.get('id', i)}]: {q['text']}")
    for rank, idx in enumerate(top_k_indices[i]):
        score = round(float(top_k_scores[i][rank]), 4)
        c = chunks[idx]
        matches.append({
            "rank": rank + 1,
            "chunk_id": c["chunk_id"],
            "scheme_name": c["scheme"],
            "section": c["section"],
            "score": score,
        })
        print(f"  #{rank+1} {c['scheme']} [{c['section']}] — {score}")
    results.append({"query_id": q.get("id"), "query": q["text"], "language": q["language"], "top_matches": matches})


Query (en) [q01]: What schemes are available for farmers with less than 2 acres of land?
  #1 pradhan mantri kisan samman nidhi (pm-kisan) [eligibility] — 0.6061
  #2 pradhan mantri fasal bima yojana (pmfby) [eligibility] — 0.6019
  #3 pradhan mantri fasal bima yojana (pmfby) [coverage_category] — 0.5481

Query (en) [q02]: Is there any scholarship for SC/ST students in college?
  #1 pradhan mantri vishwakarma [benefits] — 0.359
  #2 sukanya samriddhi account (ssa) [coverage_category] — 0.3086
  #3 stand-up india [eligibility] — 0.2972

Query (en) [q03]: What is the eligibility for the PM Awas Yojana housing scheme?
  #1 pradhan mantri awas yojana gramin (pmay-g) [eligibility] — 0.7187
  #2 pradhan mantri awas yojana urban (pmay-u 2.0) [eligibility] — 0.7029
  #3 ayushman bharat pradhan mantri jan arogya yojana (pm-jay) [eligibility] — 0.6909

Query (en) [q04]: Are there pension schemes for senior citizens above 60?
  #1 atal pension yojana (apy) [benefits] — 0.688
  #2 atal pension yo

**What to actually check here:** does q01 (farmer query) surface **PM-KISAN** chunks? Do the Hindi queries (q11–q20) correctly retrieve the right **English-language** scheme, proving cross-lingual matching works? Note 2–3 examples that worked and any that look wrong — this is what to demo to Sir.

In [14]:
with open("data/retrieval_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Saved data/retrieval_results.json — Person 3 will consume this for the RAG prompt.")

Saved data/retrieval_results.json — Person 3 will consume this for the RAG prompt.


## Summary — what to tell Sir

> "I worked on the multilingual NLP component. I compared two multilingual embedding models on English-Hindi similarity, then ran Top-3 retrieval on the real 75-chunk, 15-scheme government dataset. Cross-lingual matching works — Hindi queries correctly retrieve the right English-language scheme chunks. Output is saved to `retrieval_results.json` for the RAG integration."